In [58]:
setwd("/Users/hungm/The Francis Crick Dropbox/CaladoD/Matthew/github-repos/data-repos/Calado-SignatureDB")
library(tidyverse)
library(readxl)
library(writexl)
library(strpip)
rm(list = ls())
source("scripts/generate_signatures.R")

readmd_table <- function(url) {
    
  lines <- readLines(url)
  
  # Find the start and end lines of the table (lines with pipes)
  table_lines <- lines[str_detect(lines, "^\\|") & str_detect(lines, "\\|$")]
  table_lines <- table_lines[nchar(table_lines) > 0] # remove empty lines
  table_lines <- table_lines[!str_detect(table_lines, "^-+|^\\|[-: ]+\\|")] # remove the separator line (---)
  
  # Split each line by | and trim whitespace
  table_list <- str_split(table_lines, "\\|") %>%
    lapply(function(x) trimws(x)) %>%
    lapply(function(x) x[x != ""])  # Remove empty elements caused by leading/trailing pipes
  
  # Convert list to data frame
  df <- do.call(rbind, table_list) %>%
    as.data.frame(stringsAsFactors = FALSE)
  
  # Set first row as header
  colnames(df) <- df[1, ]
  df <- df[-1, , drop = FALSE]
  colnames(df) <- gsub(" ", "_", colnames(df))
  rownames(df) <- NULL
  return(df)
}

In [59]:
# Example usage:
url <- "README.md"
table <- readmd_table(url)
table <- table[,-ncol(table)]
table[[1]] <-paste0(table[[1]], ".gmt")
head(table)

Warning message in readLines(url):
"incomplete final line found on 'README.md'"
Warning message in (function (..., deparse.level = 1) :
"number of columns of result is not a multiple of vector length (arg 11)"


,GENE_SET_FILE,VERSION,ORG_SOURCE,CONTEXT,DESCRIPTION,SOURCE_DATA,MOL,DOI
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,SHA_MHG.gmt,v0.0.0,human-external,cancer,MHG signatures,REMoDL-B,RNA,10.1200/JCO.18.01314
2,ENNISHI_DHIT.gmt,v0.0.0,human-external,cancer,DHIT signature in BL with MYC and BCL2 rearrangements,Reddy_2017,RNA,10.1200/JCO.18.01583
3,LYMPH2CX_DLBCL.gmt,v0.0.0,human-external,cancer,NanoString Lymph2Cx DLBCL-ABC/GCB panel,Lenz_2008,RNA,10.1056/NEJMoa0802885; 10.1182/blood-2013-11-536433
4,REDDY_DLBCL.gmt,v0.0.0,human-external,cancer,DLBCL-ABC/GCB signature from Reddy et al.,Reddy_2017,RNA,10.1016/j.cell.2017.09.027
5,ATKINSO_CPC39.gmt,v0.0.0,human-internal,cancer,Common progenitor cell gene signature,Silva_2018,RNA,N/A
6,ATKINSO_CPC_CLUSTER.gmt,v0.0.0,mouse-internal,cancer,Cluster signatures,SC23423,RNA,N/A


In [60]:
dir <- c("human-external", "human-internal", "mouse-external", "mouse-internal")
sheets <- list()
for(i in seq_along(dir)){

    gmt <- list.files(paste0(dir[i],"/gmt/"), full.names = T)
    gmt.list <- list()
    for(j in seq_along(gmt)){
        df <- read_gmt(gmt[j]) %>% t() %>% as.data.frame() %>% rownames_to_column("GENE_SET_NAME")
        df$GENES <- apply(df[, 3:ncol(df)], 1, function(x) paste(na.omit(x), collapse = ", "))
        df <- df %>%
            mutate(
                GENE_SET_FILE = basename(gmt[j]),
                ORG = gsub("\\-.*", "", dir[i]),
                N = ifelse(str_detect(GENES, "[A-Z]"), 1, 0),
                N = N + str_count(GENES, ", ")) %>%
            dplyr::select(GENE_SET_NAME, GENE_SET_FILE, ORG, N, GENES)
        gmt.list[[j]] <- df}
    gmt.list <- bind_rows(gmt.list)
    
    sheets[[i]] <- merge(gmt.list, table, by = "GENE_SET_FILE", all.x = T) %>%
        dplyr::select(GENE_SET_NAME, GENE_SET_FILE, VERSION, N, ORG, ORG_SOURCE, CONTEXT, DESCRIPTION, SOURCE_DATA, MOL, DOI, GENES)
    print(head(sheets[[i]], n = 20))
    }
names(sheets) <- dir

            GENE_SET_NAME      GENE_SET_FILE VERSION    N   ORG     ORG_SOURCE
1            ENNISHI_DHIT   ENNISHI_DHIT.gmt  v0.0.0  103 human human-external
2         ENNISHI_DHIT_DN   ENNISHI_DHIT.gmt  v0.0.0   55 human human-external
3         ENNISHI_DHIT_UP   ENNISHI_DHIT.gmt  v0.0.0   47 human human-external
4      LYMPH2CX_DLBCL_ABC LYMPH2CX_DLBCL.gmt  v0.0.0    6 human human-external
5      LYMPH2CX_DLBCL_GCB LYMPH2CX_DLBCL.gmt  v0.0.0    7 human human-external
6         REDDY_DLBCL_ABC    REDDY_DLBCL.gmt  v0.0.0   10 human human-external
7         REDDY_DLBCL_GCB    REDDY_DLBCL.gmt  v0.0.0    7 human human-external
8     SHA_MHG_MYCn_vs_GCB        SHA_MHG.gmt  v0.0.0   53 human human-external
9  SHA_MHG_MYCn_vs_GCB_DN        SHA_MHG.gmt  v0.0.0   42 human human-external
10 SHA_MHG_MYCn_vs_GCB_UP        SHA_MHG.gmt  v0.0.0   10 human human-external
11    SHA_MHG_MYCr_vs_GCB        SHA_MHG.gmt  v0.0.0 1391 human human-external
12 SHA_MHG_MYCr_vs_GCB_DN        SHA_MHG.gmt  v0.0.0

In [62]:
human_index <- bind_rows(sheets[1:2])
mouse_index <- bind_rows(sheets[3:4])
any(is.na(human_index$VERSION))
any(is.na(mouse_index$VERSION))

[1] FALSE

[1] FALSE

In [63]:
write.table(human_index, file = "releases/v0.0.2/human_index.txt", sep = "\t", row.names = F, col.names = T, quote = F)
write.table(mouse_index, file = "releases/v0.0.2/mouse_index.txt", sep = "\t", row.names = F, col.names = T, quote = F)